In [0]:
# =============================================================================
# Ground Truth Data Creation and Management
# =============================================================================
# This notebook creates and manages ground truth data for model validation.
# It handles data ingestion, deduplication, and upsert operations.
# =============================================================================

# Import required libraries
from pyspark.sql.functions import (
    col,
    date_format,
    lit,
    year,
    month,
    dayofmonth,
    hour,
    row_number,
    current_timestamp,
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
)
from pyspark.sql.window import Window
from datetime import datetime
import uuid

# =============================================================================
# Configuration and Setup
# =============================================================================
# Get configuration parameters from Databricks widgets
dbutils.widgets.text("SCHEMA", "prj_yoda", "Schema")
dbutils.widgets.text("CATALOG", "ai_engineering", "Catalog")
dbutils.widgets.text("START_DATE", "2023-10-01", "Start Date")
dbutils.widgets.text("END_DATE", "2025-06-01", "End Date")
dbutils.widgets.text(
    "GROUND_TRUTH_TABLE", "ai_engineering.prj_yoda.ground_truth", "Ground Truth Table"
)
dbutils.widgets.text("PROJECT_NAME", "yoda_v2", "Project Name")

catalog = dbutils.widgets.get("CATALOG")
schema = dbutils.widgets.get("SCHEMA")
start_date = dbutils.widgets.get("START_DATE")
end_date = dbutils.widgets.get("END_DATE")
ground_truth_table = dbutils.widgets.get("GROUND_TRUTH_TABLE")
project_name = dbutils.widgets.get("PROJECT_NAME")

# Generate unique identifier for this batch
batch_uuid = uuid.uuid4().hex

print(f"🔧 Configuration:")
print(f"   Catalog: {catalog}")
print(f"   Schema: {schema}")
print(f"   Date Range: {start_date} to {end_date}")
print(f"   Ground Truth Table: {ground_truth_table}")
print(f"   Project Name: {project_name}")
print(f"   Batch UUID: {batch_uuid}")


# =============================================================================
# Helper Function: Delta Table Merge with Deduplication
# =============================================================================
def merge_with_delta_table(table_name, new_df, id_column="id"):
    """
    Merge new records into Delta table using MERGE operation to avoid duplicates.

    This function uses Delta Lake's MERGE operation to:
    1. Update existing records if ID matches (upsert)
    2. Insert new records if ID doesn't exist

    Args:
        table_name: Target Delta table name
        new_df: DataFrame with new records to merge
        id_column: Column name for unique identification

    Returns:
        Tuple of (updated_count, inserted_count)
    """
    from delta.tables import DeltaTable

    print(f"🔄 Performing MERGE operation to avoid duplicates...")

    # Get the Delta table
    delta_table = DeltaTable.forName(spark, table_name)

    # Perform merge operation (upsert)
    # - WHEN MATCHED: Update the existing record with new values
    # - WHEN NOT MATCHED: Insert the new record
    merge_result = (
        delta_table.alias("target")
        .merge(new_df.alias("source"), f"target.{id_column} = source.{id_column}")
        .whenMatchedUpdate(
            set={
                "uuid": col("source.uuid"),
                "ground_truth": col("source.ground_truth"),
                "project_name": col("source.project_name"),
                "timestamp": col("source.timestamp"),
            }
        )
        .whenNotMatchedInsert(
            values={
                "uuid": col("source.uuid"),
                "id": col("source.id"),
                "ground_truth": col("source.ground_truth"),
                "project_name": col("source.project_name"),
                "timestamp": col("source.timestamp"),
            }
        )
        .execute()
    )

    print(f"✅ MERGE operation completed")
    print(f"   This ensures no duplicate IDs in the table")

    return merge_result


# =============================================================================
# Data Processing Pipeline
# =============================================================================

# Step 1: Load data using the same SQL query as training
print(f"📚 Loading data from {catalog}.{schema} tables...")

query = f"""
WITH
daterange AS (
  SELECT
    MIN(snapshot_date) as snapshot_date,
    QUARTER(snapshot_date) as trans_quarter,
    YEAR(snapshot_date) as trans_year
  FROM {catalog}.{schema}.customer_prumdm_daily_table
  WHERE snapshot_date BETWEEN '{start_date}' AND '{end_date}'
  GROUP BY QUARTER(snapshot_date), YEAR(snapshot_date)
  ORDER BY snapshot_date
),
policy_filter AS (
  SELECT DISTINCT party_id
  FROM sdm.prumdm_enc.policy
  WHERE party_id LIKE 'LA%'
    AND (status = 'In Force' OR status = 'Paid Up Contract')
),
table_a AS (
  SELECT
    a.*,
    DATE_FORMAT(ADD_MONTHS(a.snapshot_date, 3), 'yyyy-MM') as a_join_month
  FROM {catalog}.{schema}.customer_prumdm_daily_table a
  INNER JOIN daterange d ON a.snapshot_date = d.snapshot_date
  INNER JOIN policy_filter p ON a.party_id = p.party_id
  WHERE a.party_id LIKE 'LA%'
),
table_b AS (
  SELECT
    b.party_id,
    b.ci_purchase_ind,
    b.medical_purchase_ind,
    b.protection_purchase_ind,
    b.savings_purchase_ind,
    b.investment_purchase_ind,
    b.retirement_purchase_ind,
    b.legacy_planning_purchase_ind,
    DATE_FORMAT(b.snapshot_date, 'yyyy-MM') as b_snapshot_month
  FROM {catalog}.{schema}.customer_target_yoda_daily_table b
  INNER JOIN policy_filter p ON b.party_id = p.party_id
  WHERE b.party_id LIKE 'LA%'
)
SELECT
  a.party_id,
  a.snapshot_date,
  DATE_FORMAT(a.snapshot_date, 'yyyy-MM') as trans_yyyymm,
  b.ci_purchase_ind,
  b.medical_purchase_ind,
  b.protection_purchase_ind,
  b.savings_purchase_ind,
  b.investment_purchase_ind,
  b.retirement_purchase_ind,
  b.legacy_planning_purchase_ind,
  a.*
FROM table_a a
INNER JOIN table_b b
  ON a.party_id = b.party_id
  AND a.a_join_month = b.b_snapshot_month
"""

raw_data = spark.sql(query)
initial_count = raw_data.count()
print(f"📊 Loaded {initial_count} records from SQL query")

# Step 2: Transform and prepare ground truth data
print(f"🔄 Transforming data for ground truth table...")


# Create multiclass target from purchase indicators
def create_multiclass_target(row):
    target_cols = [
        "ci_purchase_ind",
        "medical_purchase_ind",
        "protection_purchase_ind",
        "savings_purchase_ind",
        "investment_purchase_ind",
        "retirement_purchase_ind",
        "legacy_planning_purchase_ind",
    ]
    for idx, col_name in enumerate(target_cols, 1):
        if row[col_name] == 1:
            return idx
    return 0


# Register UDF
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import udf

create_target_udf = udf(create_multiclass_target, IntegerType())

ground_truth_df_initial = (
    raw_data.withColumn("uuid", lit(batch_uuid))  # Add batch identifier
    .withColumn("id", col("party_id").cast("string"))  # Use party_id as ID
    .withColumn(
        "ground_truth",
        create_target_udf(
            col("ci_purchase_ind"),
            col("medical_purchase_ind"),
            col("protection_purchase_ind"),
            col("savings_purchase_ind"),
            col("investment_purchase_ind"),
            col("retirement_purchase_ind"),
            col("legacy_planning_purchase_ind"),
        ).cast("string"),
    )  # Create multiclass target
    .withColumn("project_name", lit(project_name))  # Add project identifier
    .withColumn("timestamp", current_timestamp())  # Add processing timestamp
    .select(
        "uuid", "id", "ground_truth", "project_name", "timestamp"
    )  # Select final columns
)

initial_prepared_count = ground_truth_df_initial.count()
print(f"📊 Prepared {initial_prepared_count} records (before deduplication)")

# Step 2.5: Deduplicate source data to prevent merge conflicts
# Delta merge fails if multiple source rows match the same target row
print(f"🔍 Checking for duplicate IDs in source data...")

# Check for duplicates
duplicate_count = (
    ground_truth_df_initial.groupBy("id").count().filter(col("count") > 1).count()
)

if duplicate_count > 0:
    print(f"⚠️  Found {duplicate_count} duplicate IDs in source data - deduplicating...")

    # Deduplicate by keeping the latest record per ID (based on timestamp)
    # Using window function to rank by timestamp descending
    windowSpec = Window.partitionBy("id").orderBy(col("timestamp").desc())

    ground_truth_df = (
        ground_truth_df_initial.withColumn("row_num", row_number().over(windowSpec))
        .filter(col("row_num") == 1)  # Keep only the first (latest) record per ID
        .drop("row_num")
    )

    final_count = ground_truth_df.count()
    deduplicated_count = initial_prepared_count - final_count
    print(f"✅ Deduplication complete: removed {deduplicated_count} duplicate records")
    print(f"📊 Final count: {final_count} unique records")
else:
    print(f"✅ No duplicate IDs found in source data")
    ground_truth_df = ground_truth_df_initial
    final_count = initial_prepared_count
    print(f"📊 Prepared {final_count} records for ground truth table")

# Step 3: Write to Delta table (create or merge)
print(f"💾 Writing to ground truth table: {ground_truth_table}")

# Parse catalog, schema, and table from full table name
table_parts = ground_truth_table.split(".")
if len(table_parts) == 3:
    catalog_name, schema_name, table_name = table_parts

    # Set current catalog for Unity Catalog
    spark.sql(f"USE CATALOG {catalog_name}")
    print(f"✅ Using catalog: {catalog_name}")

    # Ensure catalog and schema exist
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
        print(f"✅ Catalog and schema verified: {catalog_name}.{schema_name}")
    except Exception as e:
        print(f"⚠️  Catalog/schema creation warning: {str(e)}")

# Display sample of data to be written
print(f"📋 Sample of ground truth data:")
ground_truth_df.show(5, truncate=False)

# Check if table exists with error handling
try:
    table_exists = spark.catalog.tableExists(ground_truth_table)
except Exception as e:
    print(f"⚠️  Could not check table existence: {str(e)}")
    table_exists = False

if table_exists:
    print("📋 Table exists - performing MERGE to prevent duplicates...")
    merge_with_delta_table(ground_truth_table, ground_truth_df, "id")

    print(f"📈 Processing Summary:")
    print(f"   - Input records: {initial_count}")
    print(f"   - Records processed: {final_count}")
    print(f"   - Operation: MERGE (upsert) - no duplicates created")

else:
    print("🆕 Table doesn't exist - creating new table...")
    print(f"💾 Writing {final_count} records in 'overwrite' mode...")
    ground_truth_df.write.format("delta").mode("overwrite").option(
        "mergeSchema", "true"
    ).option("overwriteSchema", "true").saveAsTable(ground_truth_table)

    print(f"📈 Processing Summary:")
    print(f"   - Input records: {initial_count}")
    print(f"   - Records written: {final_count}")
    print(f"✅ Ground truth table created successfully!")

# Verify write by reading back
result_count = spark.table(ground_truth_table).count()
print(f"📊 Verification: Table now contains {result_count} total records")

# Check for duplicates
duplicate_check = spark.sql(
    f"""
    SELECT id, COUNT(*) as count
    FROM {ground_truth_table}
    GROUP BY id
    HAVING COUNT(*) > 1
"""
)

duplicate_count = duplicate_check.count()
if duplicate_count > 0:
    print(f"⚠️  WARNING: Found {duplicate_count} duplicate IDs!")
    duplicate_check.show(10)
else:
    print(f"✅ No duplicate IDs found - data integrity confirmed")

print(f"🏁 Ground truth processing completed for batch: {batch_uuid}")

# Optional: Display sample of the data for verification
print(f"📋 Sample of processed data:")
spark.table(ground_truth_table).orderBy(col("timestamp").desc()).limit(5).display()